# Spark MLlib Intent Classifier: Banking77

This notebook trains a 77-class intent classifier with **Spark MLlib** and uses it to label the **unlabeled** messages in the simulated OLTP stream, following the vignette’s setup:

- messages that arrive **with** a manual intent label are training data
- messages that arrive **without** one are scored by the model

Everything is written in PySpark, so the same code runs on a laptop (`local[*]`, dev sample) and on the EC2 cluster (S3 paths, full stream). When the course’s larger synthetic Banking77 messages are released, they can be dropped in by changing `TRAIN_PATH` or `STREAM_PATH`.

Steps 1. Train on the Banking77 train split, evaluate on the held-out test split 2. Inspect accuracy for the fraud-relevant intents 3. Score every unlabeled message in the stream and save the predictions 4. Flag low-confidence predictions for human review

In [12]:
import time
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import RegexTokenizer, NGram, CountVectorizer, IDF, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# ---- Settings: the only lines to change on the cluster ----
ON_CLUSTER = False
BUCKET = "dsan6000-banking77-mh"

ROOT = Path.cwd().parent if Path.cwd().name == "spark" else Path.cwd()
if ON_CLUSTER:
    ip = !hostname -I
    MASTER = f"spark://{ip[0].split()[0]}:7077"
    TRAIN_PATH = f"s3a://{BUCKET}/banking77/processed/banking77_clean.parquet"
    DIM_PATH = f"s3a://{BUCKET}/banking77/processed/dim_intent.parquet"
    STREAM_PATH = f"s3a://{BUCKET}/banking77/compacted/messages"
    OUT = f"s3a://{BUCKET}/banking77/results"
else:
    MASTER = "local[*]"
    TRAIN_PATH = (ROOT / "data/processed/banking77_clean.parquet").as_posix()
    DIM_PATH = (ROOT / "data/processed/dim_intent.parquet").as_posix()
    STREAM_PATH = (ROOT / "data/simulated/messages").as_posix()
    OUT = (ROOT / "data/simulated/results").as_posix()      # under data/simulated, so git ignores it

LOCAL_OUT = ROOT / "spark" / "output"
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
REVIEW_THRESHOLD = 0.5          # predictions below this confidence go to a human reviewer

builder = (
    SparkSession.builder.appName("banking77-intent-classifier").master(MASTER)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "48")
)
if ON_CLUSTER:
    builder = (builder
               .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.5.0")
               .config("spark.executor.memory", "4g"))
else:
    builder = builder.config("spark.driver.memory", "4g")
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("master:", MASTER, "| cores:", spark.sparkContext.defaultParallelism)

timings = {}
def timed(name, fn):
    start = time.time()
    out = fn()
    timings[name] = round(time.time() - start, 1)
    print(f"{name}: {timings[name]} s")
    return out

26/11/26 11:15:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.

master: local[*] | cores: 8

## 1 1. Train and evaluate

In [13]:
clean = spark.read.parquet(TRAIN_PATH)
dim = spark.read.parquet(DIM_PATH).select(
    F.col("intent_id").cast("double").alias("label"), "intent", "fraud_type", "risk_level")

train = clean.where(F.col("split") == "train").select("text", F.col("label").cast("double"))
test = clean.where(F.col("split") == "test").select("text", F.col("label").cast("double"))
print(f"train: {train.count():,}  test: {test.count():,}")

# Words and word pairs, each weighted by TF-IDF, then a multinomial logistic regression.
pipeline = Pipeline(stages=[
    RegexTokenizer(inputCol="text", outputCol="tokens", pattern="[^a-z0-9']+", toLowercase=True),
    NGram(n=2, inputCol="tokens", outputCol="bigrams"),
    CountVectorizer(inputCol="tokens", outputCol="uni_tf", minDF=2),
    CountVectorizer(inputCol="bigrams", outputCol="bi_tf", minDF=2),
    IDF(inputCol="uni_tf", outputCol="uni_tfidf"),
    IDF(inputCol="bi_tf", outputCol="bi_tfidf"),
    VectorAssembler(inputCols=["uni_tfidf", "bi_tfidf"], outputCol="features"),
    LogisticRegression(featuresCol="features", labelCol="label", maxIter=100,
                       regParam=0.01, elasticNetParam=0.0, family="multinomial"),
])
model = timed("train pipeline", lambda: pipeline.fit(train))

train: 10,003  test: 3,080

train pipeline: 49.1 s

In [14]:
pred_test = model.transform(test).cache()
evaluate = lambda metric: MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName=metric).evaluate(pred_test)
metrics = {m: round(timed(f"evaluate {m}", lambda m=m: evaluate(m)), 4)
           for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]}
pd.Series(metrics, name="test split").to_frame()

26/11/26 11:16:15 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
                                                                                

evaluate accuracy: 1.0 s
evaluate f1: 0.1 s

26/11/26 11:16:15 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
26/11/26 11:16:16 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB

evaluate weightedPrecision: 0.1 s
evaluate weightedRecall: 0.1 s

26/11/26 11:16:16 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB

## 2 2. Accuracy for fraud-relevant intents

For a fraud desk the costly mistake is a fraud-relevant message (for example `compromised_card`) predicted as an ordinary intent, because it would land in a routine queue.

In [15]:
by_type = (
    pred_test.join(dim, "label")
    .join(dim.select(F.col("label").alias("prediction"), F.col("fraud_type").alias("pred_fraud_type")),
          "prediction", "left")
    .withColumn("fraud_group", F.coalesce("fraud_type", F.lit("ordinary")))
    .groupBy("fraud_group").agg(
        F.count("*").alias("test_messages"),
        F.round(F.avg((F.col("prediction") == F.col("label")).cast("int")), 3).alias("intent_accuracy"),
        # fraud-relevant messages predicted as an ordinary intent
        F.round(F.avg((F.col("fraud_type").isNotNull() & F.col("pred_fraud_type").isNull()).cast("int")), 3)
         .alias("sent_to_ordinary_queue"),
    ).orderBy("intent_accuracy")
    .toPandas()
)
by_type.to_csv(LOCAL_OUT / "intent_accuracy_by_fraud_type.csv", index=False)
by_type

26/11/26 11:16:16 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB

In [16]:
model.write().overwrite().save(f"{OUT}/intent_model")
print("model saved to", f"{OUT}/intent_model")

26/11/26 11:16:18 WARN TaskSetManager: Stage 127 contains a task of very large size (6268 KiB). The maximum recommended task size is 1000 KiB.

model saved to /Users/hanmingyang/Desktop/DSAN6000-Project/data/simulated/results/intent_model

## 3 3. Score the unlabeled stream

About 30% of stream messages arrive with a manual label and 70% without one. The saved model is reloaded (as a scoring job would) and applied only to the unlabeled messages.

In [17]:
stream = spark.read.option("basePath", STREAM_PATH).parquet(STREAM_PATH)
labeled = stream.where(F.col("label").isNotNull())
unlabeled = stream.where(F.col("label").isNull())
n_lab, n_unlab = timed("split labeled / unlabeled", lambda: (labeled.count(), unlabeled.count()))
print(f"labeled (training table): {n_lab:,}   unlabeled (scoring table): {n_unlab:,}")

split labeled / unlabeled: 0.5 s
labeled (training table): 149,964   unlabeled (scoring table): 350,036

In [18]:
scoring_model = PipelineModel.load(f"{OUT}/intent_model")

scored = (
    scoring_model.transform(unlabeled.select("message_id", "user_id", "session_id",
                                             "received_ts", "text", "date"))
    .withColumn("confidence", F.array_max(vector_to_array("probability")))
    .join(F.broadcast(dim.withColumnRenamed("label", "prediction")
                         .withColumnRenamed("intent", "predicted_intent")), "prediction", "left")
    .withColumn("needs_review", F.col("confidence") < REVIEW_THRESHOLD)
    .select("message_id", "user_id", "session_id", "received_ts", "date", "text",
            F.col("prediction").cast("int").alias("predicted_label"), "predicted_intent",
            "fraud_type", "risk_level", F.round("confidence", 4).alias("confidence"), "needs_review")
)
timed("score and write predictions",
      lambda: scored.write.mode("overwrite").partitionBy("date").parquet(f"{OUT}/predicted_intents"))

26/11/26 11:16:21 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
                                                                                

score and write predictions: 9.3 s

## 4 4. What the scoring job produced

In [19]:
preds = spark.read.parquet(f"{OUT}/predicted_intents")
summary = timed("summarise predictions", lambda: preds.agg(
    F.count("*").alias("scored_messages"),
    F.round(F.avg("confidence"), 3).alias("mean_confidence"),
    F.round(F.avg(F.col("needs_review").cast("int")), 4).alias("share_needs_review"),
    F.round(F.avg(F.col("risk_level").isin("high", "medium").cast("int")), 4).alias("share_risky_intent"),
).toPandas())
summary.to_csv(LOCAL_OUT / "intent_scoring_summary.csv", index=False)
summary.T.rename(columns={0: "value"})

summarise predictions: 0.6 s

In [20]:
top = (preds.groupBy("predicted_intent", "risk_level").count()
       .orderBy(F.desc("count")).limit(10).toPandas())
top

In [21]:
timing_table = pd.Series(timings, name="seconds").to_frame()
timing_table.to_csv(LOCAL_OUT / "intent_timings.csv")
pd.Series(metrics).to_csv(LOCAL_OUT / "intent_test_metrics.csv", header=["value"])
timing_table

In [22]:
spark.stop()